# MotionCtrl — Quaternion + SLERP vs Vanilla
รัน cell ตามลำดับครับ

# ═══════════════════════════════════════
# CONFIG — แก้ตรงนี้ที่เดียว
# ═══════════════════════════════════════
MOTIONCTRL_DIR = '/kaggle/working/MotionCtrl'   # จะ clone มาไว้ตรงนี้

# ↓↓↓ แก้ path ให้ตรงกับ Kaggle Dataset ของคุณ ↓↓↓
POSE_JSON      = '/kaggle/input/YOUR_DATASET/camera_poses.json'  # camera_poses.json จากเพื่อน (Nine)
CKPT_PATH      = '/kaggle/input/YOUR_DATASET/motionctrl.pth'     # motionctrl.pth checkpoint

ATTEMPT        = 3
OUTPUT_DIR     = f'/kaggle/working/outputs/attempt_{ATTEMPT}'
PROMPT         = 'a girl standing in front of a small wooden house, surrounded by trees, sunny day, clear sky'
N_FRAMES       = 16
HEIGHT         = 256
WIDTH          = 256
DDIM_STEPS     = 50

# ตรวจสอบว่าไฟล์อยู่ครบ
import os
assert os.path.exists(POSE_JSON), f'❌ ไม่เจอ POSE_JSON: {POSE_JSON}'
assert os.path.exists(CKPT_PATH), f'❌ ไม่เจอ CKPT_PATH: {CKPT_PATH}'
print('✅ ไฟล์ครบ พร้อมรัน')

In [ ]:
# ═══════════════════════════════════════
# CONFIG — แก้ตรงนี้ที่เดียว
# ═══════════════════════════════════════
MOTIONCTRL_DIR = '/kaggle/working/MotionCtrl'   # จะ clone มาไว้ตรงนี้
POSE_JSON      = '/kaggle/input/datasets/yasintornsun/676767/camera_poses.json'  # อัพโหลดขึ้น Kaggle Dataset
CKPT_PATH      = '/kaggle/input/datasets/yasintornsun/676767/motionctrl.pth'  # อัพโหลดขึ้น Kaggle Dataset
ATTEMPT        = 3
OUTPUT_DIR     = f'/kaggle/working/outputs/attempt_{ATTEMPT}'
PROMPT         = 'a girl standing in front of a small wooden house, surrounded by trees, sunny day, clear sky'
N_FRAMES       = 16
HEIGHT         = 256
WIDTH          = 256
DDIM_STEPS     = 50

In [ ]:
# ═══════════════════════════════════════
# CELL 1 — ติดตั้ง dependencies
# ═══════════════════════════════════════
!pip install -q setuptools==69.5.1
!pip uninstall -y pytorch-lightning lightning-fabric lightning
!pip install -q lightning==2.2.0
!pip install -q omegaconf einops open-clip-torch==2.20.0 imageio imageio-ffmpeg kornia timm

In [ ]:
# ═══════════════════════════════════════
# CELL 2 — Clone MotionCtrl repo
# ═══════════════════════════════════════
import os
if not os.path.exists(MOTIONCTRL_DIR):
    !git clone https://github.com/TencentARC/MotionCtrl {MOTIONCTRL_DIR}
else:
    print('already cloned')

os.chdir(MOTIONCTRL_DIR)
print('cwd:', os.getcwd())

In [ ]:
# ═══════════════════════════════════════
# CELL 3 — Camera math functions (จากไฟล์เพื่อน)
# ═══════════════════════════════════════
import math
import numpy as np
from dataclasses import dataclass
from typing import List

@dataclass
class Quat:
    w: float; x: float; y: float; z: float

    def __mul__(self, other):
        w1,x1,y1,z1 = self.w,self.x,self.y,self.z
        w2,x2,y2,z2 = other.w,other.x,other.y,other.z
        return Quat(w1*w2-x1*x2-y1*y2-z1*z2, w1*x2+x1*w2+y1*z2-z1*y2,
                    w1*y2-x1*z2+y1*w2+z1*x2, w1*z2+x1*y2-y1*x2+z1*w2)

    def conjugate(self): return Quat(self.w,-self.x,-self.y,-self.z)
    def dot(self, o): return self.w*o.w+self.x*o.x+self.y*o.y+self.z*o.z
    def norm(self): return math.sqrt(self.w**2+self.x**2+self.y**2+self.z**2)
    def normalized(self):
        n=self.norm(); return Quat(self.w/n,self.x/n,self.y/n,self.z/n)
    def to_list(self): return [self.w,self.x,self.y,self.z]

    @staticmethod
    def look_at(eye, center, up=np.array([0.,1.,0.])):
        forward = center-eye; forward = forward/np.linalg.norm(forward)
        if abs(np.dot(forward,up))>0.9999: up=np.array([0.,0.,1.])
        right = np.cross(forward,up); right=right/np.linalg.norm(right)
        true_up = np.cross(right,forward)
        R = np.array([[right[0],true_up[0],-forward[0]],
                      [right[1],true_up[1],-forward[1]],
                      [right[2],true_up[2],-forward[2]]])
        return Quat._matrix_to_quat(R).normalized()

    @staticmethod
    def _matrix_to_quat(R):
        trace = R[0,0]+R[1,1]+R[2,2]
        if trace>0:
            s=0.5/math.sqrt(trace+1.)
            return Quat(0.25/s,(R[2,1]-R[1,2])*s,(R[0,2]-R[2,0])*s,(R[1,0]-R[0,1])*s)
        if R[0,0]>R[1,1] and R[0,0]>R[2,2]:
            s=2.*math.sqrt(1.+R[0,0]-R[1,1]-R[2,2])
            return Quat((R[2,1]-R[1,2])/s,0.25*s,(R[0,1]+R[1,0])/s,(R[0,2]+R[2,0])/s)
        if R[1,1]>R[2,2]:
            s=2.*math.sqrt(1.+R[1,1]-R[0,0]-R[2,2])
            return Quat((R[0,2]-R[2,0])/s,(R[0,1]+R[1,0])/s,0.25*s,(R[1,2]+R[2,1])/s)
        s=2.*math.sqrt(1.+R[2,2]-R[0,0]-R[1,1])
        return Quat((R[1,0]-R[0,1])/s,(R[0,2]+R[2,0])/s,(R[1,2]+R[2,1])/s,0.25*s)

def slerp(q1, q2, t):
    dot = q1.dot(q2)
    if dot<0.: q2=Quat(-q2.w,-q2.x,-q2.y,-q2.z); dot=-dot
    dot=min(dot,1.)
    if dot>1.-1e-4:
        return Quat(q1.w+t*(q2.w-q1.w),q1.x+t*(q2.x-q1.x),
                    q1.y+t*(q2.y-q1.y),q1.z+t*(q2.z-q1.z)).normalized()
    omega=math.acos(dot); sin_omega=math.sin(omega)
    k1=math.sin((1.-t)*omega)/sin_omega; k2=math.sin(t*omega)/sin_omega
    return Quat(k1*q1.w+k2*q2.w,k1*q1.x+k2*q2.x,k1*q1.y+k2*q2.y,k1*q1.z+k2*q2.z)

def quat_to_rotmat(q):
    if isinstance(q, Quat): w,x,y,z = q.w,q.x,q.y,q.z
    else:
        q=np.array(q)/np.linalg.norm(q); w,x,y,z=q
    return np.array([[1-2*(y*y+z*z),2*(x*y-w*z),2*(x*z+w*y)],
                     [2*(x*y+w*z),1-2*(x*x+z*z),2*(y*z-w*x)],
                     [2*(x*z-w*y),2*(y*z+w*x),1-2*(x*x+y*y)]])

def orbital_path(center=[0.,0.,0.], radius=5., n_frames=16,
                 elevation_deg=20., azimuth_offset_deg=0.):
    center_arr = np.asarray(center, dtype=float)
    elev_rad   = math.radians(elevation_deg)
    azim0_rad  = math.radians(azimuth_offset_deg)
    world_up   = np.array([0.,0.,1.] if abs(elevation_deg)>85. else [0.,1.,0.])
    poses = []
    for i in range(n_frames):
        phi = azim0_rad + 2.*math.pi*i/n_frames
        x   = center_arr[0]+radius*math.cos(elev_rad)*math.cos(phi)
        y   = center_arr[1]+radius*math.sin(elev_rad)
        z   = center_arr[2]+radius*math.cos(elev_rad)*math.sin(phi)
        eye = np.array([x,y,z])
        q   = Quat.look_at(eye, center_arr, world_up)
        poses.append({"frame":i,"q":q.to_list(),"pos":[x,y,z]})
    return poses

print('✅ Camera math functions ready')

In [ ]:
# ═══════════════════════════════════════
# CELL 4 — Generate 16-frame orbit โดยตรง
# ═══════════════════════════════════════
import json

def make_RT_orbit(n_frames=16, radius=5., elevation_deg=20.):
    poses = orbital_path(n_frames=n_frames, radius=radius, elevation_deg=elevation_deg)
    RT_list = []
    for d in poses:
        R = quat_to_rotmat(Quat(*d['q']))
        t = np.array(d['pos'])
        RT_list.append(np.hstack([R, t.reshape(3,1)]))
    return np.array(RT_list)

# no slerp = เลือก 16 frames แรกจาก 120 (หมุนแค่นิดเดียว)
with open(POSE_JSON, 'r') as f:
    raw = json.load(f)
raw = sorted(raw, key=lambda x: x['frame'])

RT_no_slerp = np.array([
    np.hstack([quat_to_rotmat(np.array(d['q'])), np.array(d['pos']).reshape(3,1)])
    for d in raw[:16]
])

# slerp = generate 16 frames ครบ 360° ตรงๆ จาก math
RT_slerp = make_RT_orbit(n_frames=16, radius=5., elevation_deg=20.)

print(f'RT_no_slerp: {RT_no_slerp.shape}')
print(f'RT_slerp:    {RT_slerp.shape}')

In [ ]:
# ═══════════════════════════════════════
# CELL 5 — Normalize scale + save JSON
# ═══════════════════════════════════════
def normalize_and_save(RT_array, filename):
    RT = RT_array.copy()
    max_pos = np.abs(RT[:, :, 3]).max()
    RT[:, :, 3] = RT[:, :, 3] / max_pos * 1.5

    data = [rt.flatten().tolist() for rt in RT]

    save_dir = os.path.join(MOTIONCTRL_DIR, 'dataset', 'camera_poses', 'basic')
    os.makedirs(save_dir, exist_ok=True)
    path = os.path.join(save_dir, filename)

    with open(path, 'w') as f:
        json.dump(data, f, indent=4)
    print(f'✅ Saved → {path}')
    return path

normalize_and_save(RT_no_slerp, 'test_camera_orbit_no_slerp.json')
normalize_and_save(RT_slerp,    'test_camera_orbit_slerp.json')

In [ ]:
# ═══════════════════════════════════════
# CELL 6 — แก้ prompt file
# ═══════════════════════════════════════
prompt_file = os.path.join(MOTIONCTRL_DIR, 'main', 'evaluation', 'motionctrl_prompts_camerapose_trajs.py')

new_content = f'''
cmcm_prompt_camerapose = {{
    'prompts': [
        '{PROMPT}',
        '{PROMPT}',
        '{PROMPT}',
    ],
    'camera_poses': [
        'test_camera_SPIN-CW-60',
        'test_camera_orbit_no_slerp',
        'test_camera_orbit_slerp',
    ]
}}

omom_prompt_traj = {{'prompts': [], 'trajs': []}}
both_prompt_camerapose_traj = {{'prompts': ['placeholder'], 'camera_poses': ['test_camera_O'], 'trajs': ['shaking_10']}}
'''

with open(prompt_file, 'w') as f:
    f.write(new_content)
print('✅ Prompt file updated')

In [ ]:
# ═══════════════════════════════════════
# CELL 7 — แก้ load_camera_pose ให้หาไฟล์ได้
# ═══════════════════════════════════════
inference_file = os.path.join(MOTIONCTRL_DIR, 'main', 'evaluation', 'motionctrl_inference.py')

with open(inference_file, 'r') as f:
    content = f.read()

old = "pose_file = [f'{cond_dir}/camera_poses/{pose}.json' for pose in camera_poses]"
new = """pose_file = []
    for pose in camera_poses:
        if os.path.exists(f'{cond_dir}/camera_poses/realestate10k/{pose}.json'):
            pose_file.append(f'{cond_dir}/camera_poses/realestate10k/{pose}.json')
        elif os.path.exists(f'{cond_dir}/camera_poses/basic/{pose}.json'):
            pose_file.append(f'{cond_dir}/camera_poses/basic/{pose}.json')
        else:
            pose_file.append(f'{cond_dir}/camera_poses/{pose}.json')"""

if old in content:
    content = content.replace(old, new)
    with open(inference_file, 'w') as f:
        f.write(content)
    print('✅ inference.py patched')
else:
    print('⚠️ already patched or pattern not found')

In [ ]:
# ═══════════════════════════════════════
# CELL 8 — แก้ save_results ให้ save เป็น gif
# ═══════════════════════════════════════
with open(inference_file, 'r') as f:
    content = f.read()

old_save = "torchvision.io.write_video(path, grid, fps=fps, video_codec='h264', options={'crf': '10'})"
new_save = """import imageio
        frames = grid.numpy().astype('uint8')
        imageio.mimwrite(path.replace('.mp4', '.gif'), [frames[i] for i in range(frames.shape[0])], fps=fps)"""

if old_save in content:
    content = content.replace(old_save, new_save)
    with open(inference_file, 'w') as f:
        f.write(content)
    print('✅ save_results patched to gif')
else:
    print('⚠️ already patched or pattern not found')

In [ ]:
# patch seed_everything + ddpm3d.py
import os

# 1. patch inference.py - แทน seed_everything
inference_file = f'{MOTIONCTRL_DIR}/main/evaluation/motionctrl_inference.py'
with open(inference_file, 'r') as f:
    content = f.read()

for old, new in [
    ('from pytorch_lightning import seed_everything',
     '''def seed_everything(seed):
    import random, numpy as np, torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)'''),
    ('from lightning_fabric.utilities.seed import seed_everything',
     '''def seed_everything(seed):
    import random, numpy as np, torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)'''),
]:
    if old in content:
        content = content.replace(old, new)
        print(f'✅ patched: {old[:40]}...')

with open(inference_file, 'w') as f:
    f.write(content)

# 2. patch ddpm3d.py - แทน pytorch_lightning ทั้งหมด
ddpm_file = f'{MOTIONCTRL_DIR}/lvdm/models/ddpm3d.py'
with open(ddpm_file, 'r') as f:
    content = f.read()

for old, new in [
    ('import pytorch_lightning as pl',
     '''import torch
class _pl_stub:
    class LightningModule(torch.nn.Module): pass
    class Callback: pass
pl = _pl_stub()'''),
    ('from pytorch_lightning.utilities import rank_zero_only',
     'def rank_zero_only(fn): return fn'),
]:
    if old in content:
        content = content.replace(old, new)
        print(f'✅ patched: {old[:40]}...')

with open(ddpm_file, 'w') as f:
    f.write(content)

print('✅ all patches done')

In [ ]:
ddpm_file = f'{MOTIONCTRL_DIR}/lvdm/models/ddpm3d.py'
with open(ddpm_file, 'r') as f:
    content = f.read()

old = '''import torch.nn as _nn
class _pl_stub:
    class LightningModule(_nn.Module):
        def __init__(self): super().__init__()
    class Callback: pass
pl = _pl_stub()'''

new = '''import torch.nn as _nn
import torch as _torch
class _pl_stub:
    class LightningModule(_nn.Module):
        def __init__(self): super().__init__()
        @property
        def device(self):
            return next(self.parameters()).device
    class Callback: pass
pl = _pl_stub()'''

if old in content:
    content = content.replace(old, new)
    with open(ddpm_file, 'w') as f:
        f.write(content)
    print('✅ patched device property')
else:
    print('⚠️ not found - ลอง print content ดูก่อน')

In [ ]:
ddim_file = f'{MOTIONCTRL_DIR}/lvdm/models/samplers/ddim.py'
with open(ddim_file, 'r') as f:
    content = f.read()

old = 'to(self.model.device)'
new = 'to(next(self.model.parameters()).device)'

if old in content:
    content = content.replace(old, new)
    with open(ddim_file, 'w') as f:
        f.write(content)
    print('✅ patched ddim.py')
else:
    print('⚠️ not found')

In [ ]:
# ═══════════════════════════════════════
# CELL 9 — Run inference!
# ═══════════════════════════════════════
import subprocess
os.makedirs(OUTPUT_DIR, exist_ok=True)

cmd = [
    'python', 'main/evaluation/motionctrl_inference.py',
    '--base',      'configs/inference/config_both.yaml',
    '--ckpt_path', CKPT_PATH,
    '--condtype',  'camera_motion',
    '--cond_dir',  'dataset',
    '--savedir',   OUTPUT_DIR,
    '--n_samples', '1',
    '--bs',        '1',
    '--ddim_steps', str(DDIM_STEPS),
    '--prompt_dir', '.',
    '--height',    str(HEIGHT),
    '--width',     str(WIDTH),
    '--seed',      '42',
]

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, cwd=MOTIONCTRL_DIR)
print('Done! exit code:', result.returncode)

In [ ]:
# ═══════════════════════════════════════
# CELL 10 — แสดงผล เปรียบเทียบ
# ═══════════════════════════════════════
from IPython.display import Image, display
import glob

gifs = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.gif')))
print(f'Found {len(gifs)} gif files')

for gif in gifs:
    print(f'\n--- {os.path.basename(gif)} ---')
    display(Image(filename=gif))